# Strategy Decision Execution Replay

This notebook feeds actual quote decisions from `inventory_quote_model.ipynb` into the event-level passive execution replay.

The previous replay quoted both sides mechanically. This replay asks the next production question: when the model chooses a side and distance, does that quote actually fill under realistic event timing, and is the fill quality still acceptable?

## Process

1. Run the inventory quote model in a clean namespace and collect its walk-forward quote decisions.
2. Select quote policies, for example inventory-score positive or inventory-score positive plus standalone-score positive.
3. Convert each model bar timestamp into a live order timestamp: bar start plus bar length plus forecast/send latency.
4. Price the quote from the live event mid and the model-selected quote distance.
5. Replay top-of-book crosses and trade-through fills over the order TTL plus cancel acknowledgement latency.
6. Measure fill rate, markout, maker rebate impact, and model-sim versus event-replay disagreement.

This notebook uses top-of-book quotes and trades. Quotes away from the touch have unknown queue position without full L2 reconstruction, so trade-through fills are optimistic for queue priority and conservative for capacity analysis when no trade-through happens.

## Execution Timing

The feature bar indexed by `t` covers the interval `[t,t+\Delta]`. The earliest live order time is therefore:

$$
t^{live}=t+\Delta+\ell_{send}
$$

The order rests until:

$$
t^{dead}=t^{live}+TTL+\ell_{cancel}
$$

For selected distance `d`, the live quote price is anchored to the live event mid `M_{live}`:

$$
Q_{bid}=M_{live}\exp(-d/10^4),\qquad Q_{ask}=M_{live}\exp(d/10^4)
$$

A full trade-through fill requires opposite-side taker volume at or through the quote price to reach the configured order quantity before cancellation.

In [ ]:
from __future__ import annotations

import json
import os
from datetime import timezone
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

ROOT = Path(os.environ.get("MODL_WS_NORMALIZED_DIR", "/mnt/burner-archive/ws_normalized")).expanduser()
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
INVENTORY_NOTEBOOK = Path(os.environ.get("MODL_INVENTORY_NOTEBOOK", "notebooks/inventory_quote_model.ipynb"))

BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))
MARKOUT_MINUTES = int(os.environ.get("MODL_STRATEGY_REPLAY_MARKOUT_MINUTES", "30"))
ORDER_QTY = float(os.environ.get("MODL_STRATEGY_REPLAY_ORDER_QTY", "0.01"))
LATENCY_GRID_MS = tuple(int(value) for value in os.environ.get("MODL_STRATEGY_REPLAY_LATENCY_MS", "200,500,1000").split(","))
TTL_GRID_MS = tuple(int(value) for value in os.environ.get("MODL_STRATEGY_REPLAY_TTL_MS", "15000,60000,300000,900000,1800000").split(","))
REBATE_GRID_BPS = tuple(float(value) for value in os.environ.get("MODL_STRATEGY_REPLAY_REBATE_BPS", "0,2.5").split(","))
CANCEL_ACK_LATENCY_MS = int(os.environ.get("MODL_STRATEGY_REPLAY_CANCEL_ACK_MS", "100"))
BASE_LATENCY_MS = int(os.environ.get("MODL_STRATEGY_REPLAY_BASE_LATENCY_MS", "200"))
BASE_TTL_MS = int(os.environ.get("MODL_STRATEGY_REPLAY_BASE_TTL_MS", "15000"))
BASE_REBATE_BPS = float(os.environ.get("MODL_STRATEGY_REPLAY_BASE_REBATE_BPS", "2.5"))
STANDALONE_FLOOR_BPS = float(os.environ.get("MODL_STRATEGY_REPLAY_STANDALONE_FLOOR_BPS", "0.0"))
SAVE_OUTPUTS = False

BAR_MS = BAR_MINUTES * 60_000
MARKOUT_MS = MARKOUT_MINUTES * 60_000

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 260)
pd.set_option("display.max_colwidth", 220)

ROOT, INVENTORY_NOTEBOOK, BAR_MINUTES, ORDER_QTY

## Generate Inventory Quote Decisions

We execute the inventory quote notebook up to the policy tables so this replay uses the same feature policy, fair-value model, fill model, toxicity residual model, inventory penalty, and model-risk haircut.

In [ ]:
def quiet_display(*args, **kwargs):
    return None


def execute_inventory_notebook(path: Path) -> dict[str, object]:
    nb = json.loads(path.read_text())
    namespace: dict[str, object] = {"__name__": "__main__", "display": quiet_display}
    skip_cells = {"diagnostic-plots", "optional-save"}
    for idx, cell in enumerate(nb["cells"]):
        if cell.get("cell_type") != "code" or cell.get("id") in skip_cells:
            continue
        namespace["display"] = quiet_display
        source = "".join(cell.get("source", []))
        exec(compile(source, f"{path}:{idx}", "exec"), namespace)
        namespace["display"] = quiet_display
        plt.close("all")
    return namespace


inventory_ns = execute_inventory_notebook(INVENTORY_NOTEBOOK)
best_inventory_quotes = inventory_ns["best_inventory_quotes"].copy()
walk_forward_summary = inventory_ns["walk_forward_summary"].copy()
inventory_summary = inventory_ns["inventory_summary"].copy()

decision_inventory = best_inventory_quotes[best_inventory_quotes["inventory_units"] == 0.0].copy()
policy_inputs = {
    "inventory_score": decision_inventory[decision_inventory["quoteable"]].copy(),
    "inventory_and_standalone": decision_inventory[
        decision_inventory["quoteable"] & (decision_inventory["predicted_standalone_value_bps"] >= STANDALONE_FLOOR_BPS)
    ].copy(),
    "bid_inventory_score": decision_inventory[decision_inventory["quoteable"] & decision_inventory["side"].eq("bid")].copy(),
}

policy_input_summary = pd.DataFrame(
    [
        {
            "policy": name,
            "decisions": len(frame),
            "bid_decisions": int(frame["side"].eq("bid").sum()) if not frame.empty else 0,
            "ask_decisions": int(frame["side"].eq("ask").sum()) if not frame.empty else 0,
            "mean_distance_bps": frame["quote_distance_bps"].mean() if not frame.empty else np.nan,
            "mean_model_quote_score_bps": frame["quote_score_bps"].mean() if not frame.empty else np.nan,
            "mean_predicted_standalone_bps": frame["predicted_standalone_value_bps"].mean() if not frame.empty else np.nan,
            "bar_label_touch_rate": frame["touch"].mean() if not frame.empty else np.nan,
            "bar_label_realized_ev_bps": frame["realized_quote_value_bps"].mean() if not frame.empty else np.nan,
        }
        for name, frame in policy_inputs.items()
    ]
)

display(walk_forward_summary)
display(policy_input_summary)
display(decision_inventory.head(20))

## Load Event Data

The replay uses local receive timestamps. If exchange timestamps are not on the same clock, they are not used for latency analysis.

In [ ]:
def load_hibachi_quotes(root: Path) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for path in sorted((root / "hibachi" / "btc_usdt-p" / "quotes").glob("*.parquet")):
        frame = (
            pl.scan_parquet(path)
            .select(
                [
                    pl.col("received_mts").cast(pl.Int64),
                    pl.col("bid_price").cast(pl.Float64, strict=False).alias("bid"),
                    pl.col("ask_price").cast(pl.Float64, strict=False).alias("ask"),
                    pl.col("bid_size").cast(pl.Float64, strict=False).alias("bid_size"),
                    pl.col("ask_size").cast(pl.Float64, strict=False).alias("ask_size"),
                ]
            )
            .collect()
            .to_pandas()
        )
        frames.append(frame)
    quotes = pd.concat(frames, ignore_index=True).dropna(subset=["received_mts", "bid", "ask"])
    quotes = quotes.sort_values("received_mts").drop_duplicates("received_mts", keep="last").reset_index(drop=True)
    quotes["mid"] = (quotes["bid"] + quotes["ask"]) / 2.0
    quotes["spread_bps"] = (quotes["ask"] - quotes["bid"]) / quotes["mid"] * 10_000.0
    return quotes


def load_hibachi_trades(root: Path) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for path in sorted((root / "hibachi" / "btc_usdt-p" / "trades").glob("*.parquet")):
        frame = (
            pl.scan_parquet(path)
            .select(
                [
                    pl.col("received_mts").cast(pl.Int64),
                    pl.col("taker_side").cast(pl.Utf8).str.to_lowercase().alias("taker_side"),
                    pl.col("price").cast(pl.Float64, strict=False).alias("price"),
                    pl.col("quantity").cast(pl.Float64, strict=False).alias("qty"),
                ]
            )
            .collect()
            .to_pandas()
        )
        frames.append(frame)
    trades = pd.concat(frames, ignore_index=True).dropna(subset=["received_mts", "taker_side", "price", "qty"])
    return trades.sort_values("received_mts").reset_index(drop=True)


quotes = load_hibachi_quotes(ROOT)
trades = load_hibachi_trades(ROOT)
event_summary = pd.DataFrame(
    [
        {
            "quote_rows": len(quotes),
            "trade_rows": len(trades),
            "first_quote": pd.to_datetime(quotes["received_mts"].min(), unit="ms", utc=True),
            "last_quote": pd.to_datetime(quotes["received_mts"].max(), unit="ms", utc=True),
            "median_spread_bps": quotes["spread_bps"].median(),
            "p95_spread_bps": quotes["spread_bps"].quantile(0.95),
        }
    ]
)
display(event_summary)

## Strategy Replay Functions

The fill rule is trade-through with no modeled queue ahead. That is appropriate for proving a selected quote did not even trade through; it is optimistic when trade-through volume exists because full L2 queue at the selected price is not reconstructed here.

In [ ]:
quote_ts = quotes["received_mts"].to_numpy(dtype=np.int64)
quote_bid = quotes["bid"].to_numpy(dtype=float)
quote_ask = quotes["ask"].to_numpy(dtype=float)
quote_mid = quotes["mid"].to_numpy(dtype=float)

trade_ts = trades["received_mts"].to_numpy(dtype=np.int64)
trade_side = trades["taker_side"].astype(str).to_numpy()
trade_price = trades["price"].to_numpy(dtype=float)
trade_qty = trades["qty"].to_numpy(dtype=float)


def timestamp_to_mts(ts: pd.Timestamp) -> int:
    timestamp = pd.Timestamp(ts)
    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize("UTC")
    else:
        timestamp = timestamp.tz_convert("UTC")
    return int(timestamp.timestamp() * 1_000)


def asof_quote_index(mts: int) -> int:
    idx = int(np.searchsorted(quote_ts, mts, side="right") - 1)
    if idx < 0 or idx >= len(quote_ts):
        return -1
    return idx


def quote_price_from_live_mid(side: str, distance_bps: float, live_idx: int) -> float:
    if side == "bid":
        return float(quote_mid[live_idx] * np.exp(-distance_bps / 10_000.0))
    return float(quote_mid[live_idx] * np.exp(distance_bps / 10_000.0))


def first_cross(side: str, price: float, start_mts: int, end_mts: int) -> float:
    left = int(np.searchsorted(quote_ts, start_mts, side="left"))
    right = int(np.searchsorted(quote_ts, end_mts, side="right"))
    if left >= right:
        return np.nan
    if side == "bid":
        condition = quote_ask[left:right] <= price
    else:
        condition = quote_bid[left:right] >= price
    hits = np.flatnonzero(condition)
    return float(quote_ts[left + hits[0]]) if len(hits) else np.nan


def first_trade_through_fill(side: str, price: float, start_mts: int, end_mts: int) -> tuple[float, float]:
    left = int(np.searchsorted(trade_ts, start_mts, side="left"))
    right = int(np.searchsorted(trade_ts, end_mts, side="right"))
    if left >= right:
        return np.nan, 0.0
    if side == "bid":
        condition = (trade_side[left:right] == "sell") & (trade_price[left:right] <= price)
    else:
        condition = (trade_side[left:right] == "buy") & (trade_price[left:right] >= price)
    candidate_offsets = np.flatnonzero(condition)
    if not len(candidate_offsets):
        return np.nan, 0.0
    cumulative_qty = 0.0
    last_mts = np.nan
    for offset in candidate_offsets:
        cumulative_qty += trade_qty[left + offset]
        last_mts = float(trade_ts[left + offset])
        if cumulative_qty >= ORDER_QTY:
            return last_mts, ORDER_QTY
    return last_mts, min(cumulative_qty, ORDER_QTY)


def execution_markout_bps(side: str, price: float, fill_mts: float) -> float:
    idx = asof_quote_index(int(fill_mts + MARKOUT_MS))
    if idx < 0:
        return np.nan
    if side == "bid":
        return float(10_000.0 * np.log(quote_mid[idx] / price))
    return float(10_000.0 * np.log(price / quote_mid[idx]))


def replay_policy(policy_name: str, decisions: pd.DataFrame, latency_ms: int, ttl_ms: int, maker_rebate_bps: float) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for decision in decisions.itertuples(index=False):
        bar_start_mts = timestamp_to_mts(decision.minute)
        decision_mts = bar_start_mts + BAR_MS
        live_mts = decision_mts + latency_ms
        live_idx = asof_quote_index(live_mts)
        if live_idx < 0:
            continue
        side = str(decision.side)
        distance_bps = float(decision.quote_distance_bps)
        order_price = quote_price_from_live_mid(side, distance_bps, live_idx)
        end_mts = live_mts + ttl_ms + CANCEL_ACK_LATENCY_MS

        cross_mts = first_cross(side, order_price, live_mts, end_mts)
        fill_mts, fill_qty = first_trade_through_fill(side, order_price, live_mts, end_mts)
        full_fill = bool(np.isfinite(fill_mts) and fill_qty >= ORDER_QTY)
        markout = execution_markout_bps(side, order_price, fill_mts) if full_fill else np.nan
        value = markout + maker_rebate_bps if np.isfinite(markout) else 0.0
        if side == "bid":
            distance_from_touch_bps = (quote_bid[live_idx] - order_price) / quote_mid[live_idx] * 10_000.0
        else:
            distance_from_touch_bps = (order_price - quote_ask[live_idx]) / quote_mid[live_idx] * 10_000.0

        rows.append(
            {
                "policy": policy_name,
                "date": decision.date,
                "minute": decision.minute,
                "side": side,
                "quote_distance_bps": distance_bps,
                "model_quote_score_bps": float(decision.quote_score_bps),
                "predicted_standalone_value_bps": float(decision.predicted_standalone_value_bps),
                "bar_label_touch": float(decision.touch),
                "bar_label_markout_bps": float(decision.markout_bps),
                "bar_label_realized_ev_bps": float(decision.realized_quote_value_bps),
                "latency_ms": latency_ms,
                "ttl_ms": ttl_ms,
                "maker_rebate_bps": maker_rebate_bps,
                "decision_mts": decision_mts,
                "live_mts": live_mts,
                "live_mid": float(quote_mid[live_idx]),
                "live_bid": float(quote_bid[live_idx]),
                "live_ask": float(quote_ask[live_idx]),
                "order_price": order_price,
                "distance_from_touch_bps": distance_from_touch_bps,
                "cross_fill": np.isfinite(cross_mts),
                "event_full_fill": full_fill,
                "event_partial_qty": fill_qty,
                "event_fill_mts": fill_mts,
                "event_fill_delay_ms": fill_mts - live_mts if np.isfinite(fill_mts) else np.nan,
                "event_markout_bps": markout,
                "event_value_before_rebate_bps": markout if np.isfinite(markout) else 0.0,
                "event_value_after_rebate_bps": value,
            }
        )
    return pd.DataFrame(rows)


def summarize_replay(frame: pd.DataFrame) -> dict[str, float]:
    filled = frame[frame["event_full_fill"]]
    return {
        "orders": int(len(frame)),
        "bar_touch_rate": float(frame["bar_label_touch"].mean()) if len(frame) else np.nan,
        "cross_fill_rate": float(frame["cross_fill"].mean()) if len(frame) else np.nan,
        "event_full_fill_rate": float(frame["event_full_fill"].mean()) if len(frame) else np.nan,
        "event_partial_rate": float((frame["event_partial_qty"] > 0).mean()) if len(frame) else np.nan,
        "event_fills": int(frame["event_full_fill"].sum()) if len(frame) else 0,
        "mean_distance_bps": float(frame["quote_distance_bps"].mean()) if len(frame) else np.nan,
        "mean_distance_from_touch_bps": float(frame["distance_from_touch_bps"].mean()) if len(frame) else np.nan,
        "mean_model_quote_score_bps": float(frame["model_quote_score_bps"].mean()) if len(frame) else np.nan,
        "mean_predicted_standalone_bps": float(frame["predicted_standalone_value_bps"].mean()) if len(frame) else np.nan,
        "mean_event_markout_if_filled_bps": float(filled["event_markout_bps"].mean()) if len(filled) else np.nan,
        "event_ev_before_rebate_bps": float(frame["event_value_before_rebate_bps"].mean()) if len(frame) else np.nan,
        "event_ev_after_rebate_bps": float(frame["event_value_after_rebate_bps"].mean()) if len(frame) else np.nan,
        "median_fill_delay_ms": float(filled["event_fill_delay_ms"].median()) if len(filled) else np.nan,
    }

## Base Strategy Replay

The base replay uses 200 ms latency, 15 second TTL, 100 ms cancel acknowledgement, and a 2.5 bps maker rebate. This is intentionally close to the neutral replay base case.

In [ ]:
base_replay_frames: list[pd.DataFrame] = []
for policy_name, decisions in policy_inputs.items():
    base_replay_frames.append(replay_policy(policy_name, decisions, BASE_LATENCY_MS, BASE_TTL_MS, BASE_REBATE_BPS))
base_strategy_replay = pd.concat(base_replay_frames, ignore_index=True)

base_strategy_summary = (
    base_strategy_replay.groupby("policy")
    .apply(lambda frame: pd.Series(summarize_replay(frame)), include_groups=False)
    .reset_index()
)
base_side_summary = (
    base_strategy_replay.groupby(["policy", "side"])
    .apply(lambda frame: pd.Series(summarize_replay(frame)), include_groups=False)
    .reset_index()
)

display(base_strategy_summary)
display(base_side_summary)
base_strategy_replay.tail(20)

## TTL, Latency, And Rebate Sensitivity

If the strategy needs very long TTLs to fill, then its quoted edge is stale for a long time. That is capacity, latency, and cancel-risk information, not just execution noise.

In [ ]:
scenario_rows: list[dict[str, object]] = []
scenario_replays: list[pd.DataFrame] = []

for policy_name, decisions in policy_inputs.items():
    for latency_ms in LATENCY_GRID_MS:
        for ttl_ms in TTL_GRID_MS:
            for maker_rebate_bps in REBATE_GRID_BPS:
                replay = replay_policy(policy_name, decisions, latency_ms, ttl_ms, maker_rebate_bps)
                summary = summarize_replay(replay)
                scenario_rows.append(
                    {
                        "policy": policy_name,
                        "latency_ms": latency_ms,
                        "ttl_ms": ttl_ms,
                        "maker_rebate_bps": maker_rebate_bps,
                        **summary,
                    }
                )
                if latency_ms == BASE_LATENCY_MS and maker_rebate_bps == BASE_REBATE_BPS:
                    scenario_replays.append(replay)

strategy_scenario_summary = pd.DataFrame(scenario_rows).sort_values("event_ev_after_rebate_bps", ascending=False).reset_index(drop=True)
strategy_replay_by_ttl = pd.concat(scenario_replays, ignore_index=True) if scenario_replays else pd.DataFrame()

display(strategy_scenario_summary.head(40))
display(
    strategy_scenario_summary[
        (strategy_scenario_summary["latency_ms"] == BASE_LATENCY_MS)
        & (strategy_scenario_summary["maker_rebate_bps"] == BASE_REBATE_BPS)
    ].sort_values(["policy", "ttl_ms"])
)

## Model Labels Versus Event Replay

This section compares bar-level model labels with event replay. A large gap means the model is learning a fill process that does not match event-level execution.

In [ ]:
model_event_gap = (
    strategy_scenario_summary[
        (strategy_scenario_summary["latency_ms"] == BASE_LATENCY_MS)
        & (strategy_scenario_summary["maker_rebate_bps"] == BASE_REBATE_BPS)
    ]
    .copy()
    .sort_values(["policy", "ttl_ms"])
)
model_event_gap["bar_minus_event_fill_rate"] = model_event_gap["bar_touch_rate"] - model_event_gap["event_full_fill_rate"]
model_event_gap["model_score_minus_event_ev_bps"] = model_event_gap["mean_model_quote_score_bps"] - model_event_gap["event_ev_after_rebate_bps"]
display(model_event_gap)

## Diagnostics

In [ ]:
base_grid = strategy_scenario_summary[
    (strategy_scenario_summary["latency_ms"] == BASE_LATENCY_MS)
    & (strategy_scenario_summary["maker_rebate_bps"] == BASE_REBATE_BPS)
].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.lineplot(data=base_grid, x="ttl_ms", y="event_full_fill_rate", hue="policy", marker="o", ax=axes[0, 0])
axes[0, 0].set_xscale("log")
axes[0, 0].set_title("Event Full Fill Rate By TTL")

sns.lineplot(data=base_grid, x="ttl_ms", y="event_ev_after_rebate_bps", hue="policy", marker="o", ax=axes[0, 1])
axes[0, 1].axhline(0, color="black", linewidth=1)
axes[0, 1].set_xscale("log")
axes[0, 1].set_title("Event EV After Rebate By TTL")

sns.barplot(data=policy_input_summary, x="policy", y="mean_distance_bps", ax=axes[1, 0])
axes[1, 0].tick_params(axis="x", rotation=20)
axes[1, 0].set_title("Selected Quote Distance")

sns.scatterplot(data=base_strategy_replay, x="model_quote_score_bps", y="predicted_standalone_value_bps", hue="side", style="policy", ax=axes[1, 1])
axes[1, 1].axhline(0, color="black", linewidth=1)
axes[1, 1].set_title("Selected Decisions: Inventory Score Versus Standalone Score")

plt.tight_layout()

## Portfolio Manager Interpretation

This replay separates two failure modes:

1. **Bad fills**: quotes fill and mark out poorly.
2. **No capacity**: quotes are so far away or so short-lived that they do not fill.

On the current walk-forward sample, a short TTL is mostly a no-capacity regime for the selected inventory quotes. That does not prove the strategy is bad, but it says the quote policy is too passive for live validation unless we accept long resting times or quote closer to the touch.

The next engineering extension is full L2 reconstruction for quote distances away from best bid/ask. The next research extension is a quote-policy constraint that optimizes expected value subject to a minimum expected fill rate.

## Optional Save

In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / "strategy_execution_replay" / f"bar_{BAR_MINUTES}m"
    out_dir.mkdir(parents=True, exist_ok=True)
    walk_forward_summary.to_parquet(out_dir / "walk_forward_summary.parquet")
    policy_input_summary.to_parquet(out_dir / "policy_input_summary.parquet")
    event_summary.to_parquet(out_dir / "event_summary.parquet")
    base_strategy_replay.to_parquet(out_dir / "base_strategy_replay.parquet")
    base_strategy_summary.to_parquet(out_dir / "base_strategy_summary.parquet")
    base_side_summary.to_parquet(out_dir / "base_side_summary.parquet")
    strategy_scenario_summary.to_parquet(out_dir / "strategy_scenario_summary.parquet")
    model_event_gap.to_parquet(out_dir / "model_event_gap.parquet")
    print(f"wrote strategy execution replay outputs to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; nothing written")